In [30]:
!nvidia-smi

Wed Apr 29 03:49:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [31]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory GB:", torch.cuda.get_device_properties(0).total_memory / 1e9)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA build: 12.8
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory GB: 85.094825984


In [32]:
from pathlib import Path
import os

repo_dir = Path("/content/modded-nanogpt")
repo_url = "https://github.com/bluepeach1121/modded-nanogpt.git"
branch = "modded-gpt-test"

if repo_dir.exists():
    print("Repo already exists. Updating existing clone.")
    os.chdir(repo_dir)
    !git fetch origin
    !git checkout {branch}
    !git pull
else:
    print("Repo not found. Cloning.")
    %cd /content
    !git clone {repo_url}
    %cd /content/modded-nanogpt
    !git checkout {branch}

print("\nCurrent branch:")
!git branch

print("\nStatus:")
!git status

Repo already exists. Updating existing clone.
Already on 'modded-gpt-test'
Your branch is up to date with 'origin/modded-gpt-test'.
Already up to date.

Current branch:
  master
* modded-gpt-test

Status:
On branch modded-gpt-test
Your branch is up to date with 'origin/modded-gpt-test'.

nothing to commit, working tree clean


In [33]:
from pathlib import Path

%cd /content/modded-nanogpt

paths = [
    "data/cached_fineweb10B.py",
    "records/track_3_optimization/train_gpt_simple.py",
    "records/track_3_optimization/README.md",
]

for p in paths:
    path = Path(p)
    print(f"{p}: {path.exists()}")

/content/modded-nanogpt
data/cached_fineweb10B.py: True
records/track_3_optimization/train_gpt_simple.py: True
records/track_3_optimization/README.md: True


In [34]:
from pathlib import Path
import shutil

%cd /content/modded-nanogpt

data_dir = Path("data/fineweb10B")

if data_dir.exists():
    print("Deleting old data folder:", data_dir.resolve())
    shutil.rmtree(data_dir)
    print("Deleted.")
else:
    print("No existing data/fineweb10B folder found.")

/content/modded-nanogpt
Deleting old data folder: /content/modded-nanogpt/data/fineweb10B
Deleted.


In [35]:
%cd /content/modded-nanogpt
!python data/cached_fineweb10B.py 40

/content/modded-nanogpt
fineweb_val_000000.bin: 100% 200M/200M [00:02<00:00, 99.0MB/s]
fineweb_train_000001.bin: 100% 200M/200M [00:02<00:00, 90.4MB/s]
fineweb_train_000002.bin: 100% 200M/200M [00:02<00:00, 76.6MB/s] 
fineweb_train_000003.bin: 100% 200M/200M [00:02<00:00, 99.4MB/s]
fineweb_train_000004.bin: 100% 200M/200M [00:02<00:00, 90.5MB/s] 
fineweb_train_000005.bin: 100% 200M/200M [00:01<00:00, 124MB/s]
fineweb_train_000006.bin: 100% 200M/200M [00:01<00:00, 142MB/s] 
fineweb_train_000007.bin: 100% 200M/200M [00:01<00:00, 124MB/s] 
fineweb_train_000008.bin: 100% 200M/200M [00:01<00:00, 110MB/s] 
fineweb_train_000009.bin: 100% 200M/200M [00:02<00:00, 76.6MB/s] 
fineweb_train_000010.bin: 100% 200M/200M [00:02<00:00, 99.4MB/s]
fineweb_train_000011.bin: 100% 200M/200M [00:01<00:00, 110MB/s] 
fineweb_train_000012.bin: 100% 200M/200M [00:01<00:00, 110MB/s] 
fineweb_train_000013.bin: 100% 200M/200M [00:01<00:00, 124MB/s] 
fineweb_train_000014.bin: 100% 200M/200M [00:02<00:00, 99.4MB/s]
f

In [36]:
from pathlib import Path

data_dir = Path("/content/modded-nanogpt/data/fineweb10B")

print("Data dir exists:", data_dir.exists())
print("Data dir:", data_dir)

if data_dir.exists():
    train_files = sorted(data_dir.glob("fineweb_train_*.bin"))
    val_files = sorted(data_dir.glob("fineweb_val_*.bin"))
    all_bin_files = sorted(data_dir.glob("*.bin"))

    total_gb = sum(f.stat().st_size for f in all_bin_files) / 1e9

    print("\nTrain shards:", len(train_files))
    print("Val shards:", len(val_files))
    print("Total .bin files:", len(all_bin_files))
    print(f"Total size: {total_gb:.3f} GB")

    if train_files:
        print("\nFirst train shard:", train_files[0].name)
        print("Last train shard:", train_files[-1].name)

    if val_files:
        print("Validation shard:", val_files[0].name)

    print("\nLast 10 train shards:")
    for f in train_files[-10:]:
        print(f"{f.name}: {f.stat().st_size / 1e9:.3f} GB")

Data dir exists: True
Data dir: /content/modded-nanogpt/data/fineweb10B

Train shards: 40
Val shards: 1
Total .bin files: 41
Total size: 8.200 GB

First train shard: fineweb_train_000001.bin
Last train shard: fineweb_train_000040.bin
Validation shard: fineweb_val_000000.bin

Last 10 train shards:
fineweb_train_000031.bin: 0.200 GB
fineweb_train_000032.bin: 0.200 GB
fineweb_train_000033.bin: 0.200 GB
fineweb_train_000034.bin: 0.200 GB
fineweb_train_000035.bin: 0.200 GB
fineweb_train_000036.bin: 0.200 GB
fineweb_train_000037.bin: 0.200 GB
fineweb_train_000038.bin: 0.200 GB
fineweb_train_000039.bin: 0.200 GB
fineweb_train_000040.bin: 0.200 GB


In [37]:
from pathlib import Path
import re

script_path = Path("/content/modded-nanogpt/records/track_3_optimization/train_gpt_simple.py")
text = script_path.read_text()

match = re.search(r"train_steps\s*=\s*(\d+)", text)
print("Script exists:", script_path.exists())
print("train_steps:", match.group(1) if match else "not found")

print("\nMuon settings lines:")
for line in text.splitlines():
    if "optimizer2 = Muon" in line or "lr=0.025" in line or "weight_decay=0.0125" in line:
        print(line)

Script exists: True
train_steps: 3500

Muon settings lines:
optimizer2 = Muon([p for p in model.blocks.parameters() if p.ndim >= 2],
                  lr=0.025, weight_decay=0.0125)


In [38]:
%cd /content/modded-nanogpt
!torchrun --standalone --nproc_per_node=1 records/track_3_optimization/smoke_train_gpt_simple.py

/content/modded-nanogpt
logs/d992fa58-ef21-4574-89a9-e48da724b5dd.txt
step:0/50 val_loss:10.82583 train_time:0.000s step_avg:0.07ms
step:1/50 train_time:5.729s step_avg:5728.94ms
step:2/50 train_time:8.264s step_avg:4132.24ms
step:3/50 train_time:10.732s step_avg:3577.28ms
step:4/50 train_time:13.199s step_avg:3299.86ms
step:5/50 train_time:15.664s step_avg:3132.87ms
step:6/50 train_time:18.130s step_avg:3021.65ms
step:7/50 train_time:20.597s step_avg:2942.36ms
step:8/50 train_time:23.061s step_avg:2882.65ms
step:9/50 train_time:25.528s step_avg:2836.40ms
step:10/50 train_time:27.992s step_avg:2799.17ms
step:11/50 train_time:30.462s step_avg:2769.30ms
step:12/50 train_time:32.928s step_avg:2744.01ms
step:13/50 train_time:35.399s step_avg:2722.97ms
step:14/50 train_time:37.866s step_avg:2704.72ms
step:15/50 train_time:40.330s step_avg:2688.67ms
step:16/50 train_time:42.794s step_avg:2674.66ms
step:17/50 train_time:45.259s step_avg:2662.30ms
step:18/50 train_time:47.726s step_avg:2651.44